# Marine Heatwave Detection — Gulf of Tunis (2016–2025)

This notebook detects and analyses **marine heatwave (MHW) events** in the Gulf of Tunis over ten summer seasons (JAS: July–August–September, 2016–2025).

**Definition used:** a pixel is classified as *in heatwave* on day *t* if its sea surface temperature has been ≥ 27.5 °C for at least 5 consecutive days ending on day *t* (i.e., the 5-day trailing rolling minimum ≥ threshold).  An *event* is a contiguous temporal period in which at least one pixel satisfies this condition.

| Step | Description |
|------|-------------|
| 1 | Load daily SST from CMEMS reprocessed L3S (Gulf of Tunis bbox) |
| 2 | Detect heatwave pixels — rolling-minimum approach |
| 3 | Label and characterise events |
| 4 | Interactive statistics plots (hvplot) |
| 5 | SST map of the 3rd event (peak day) |
| 6 | Chlorophyll-a map for the same date (Sentinel-3 OLCI via Planetary Computer) |

---
## 0. Setup

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import copernicusmarine
import pystac_client
import planetary_computer
import hvplot.xarray
import hvplot.pandas
import panel as pn
pn.extension()

# Area of interest: Gulf of Tunis  [lon_min, lat_min, lon_max, lat_max]
bbox = [10.0, 36.6, 10.9, 37.2]

# Marine heatwave parameters
THRESHOLD = 27.8   # °C
MIN_DAYS  = 5      # consecutive days
MONTHS    = [7, 8, 9]              # JAS
YEARS     = list(range(2016, 2026))

print(f"Gulf of Tunis bbox : lon [{bbox[0]}, {bbox[2]}]  lat [{bbox[1]}, {bbox[3]}]")
print(f"MHW threshold      : >= {THRESHOLD} °C for >= {MIN_DAYS} consecutive days")
print(f"Season / years     : JAS  {YEARS[0]}-{YEARS[-1]}")

Gulf of Tunis bbox : lon [10.0, 10.9]  lat [36.6, 37.2]
MHW threshold      : >= 27.8 °C for >= 5 consecutive days
Season / years     : JAS  2016-2025


---
## 1. Load CMEMS SST (JAS 2016–2025)

We use the **Mediterranean Sea SST L3S multi-year reprocessed** product, which provides daily gap-filled satellite-composite SST at ~0.01° resolution.

- **Dataset ID:** `cmems_SST_MED_PHY_L3S_MY_010_042_202211_P1D-m`
- **Variable:** `adjusted_sea_surface_temperature` (Kelvin)

> If the dataset ID raises a `ServiceNotFound` error, run `copernicusmarine.describe(contains=["SST_MED", "L3S", "MY"])` to find the current name.

In [3]:
DATASET_ID = "SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b"
VAR_NAME   = "adjusted_sea_surface_temperature"   # Kelvin

ds_sst = copernicusmarine.open_dataset(
    DATASET_ID,
    variables=[VAR_NAME],
    minimum_longitude=bbox[0],
    maximum_longitude=bbox[2],
    minimum_latitude=bbox[1],
    maximum_latitude=bbox[3],
    start_datetime="2016-07-01T00:00:00",
    end_datetime="2025-09-30T23:59:59",
)
print(f"Dataset dims : {dict(ds_sst.sizes)}")
print(f"Time range   : {str(ds_sst.time.values[0])[:10]} -> {str(ds_sst.time.values[-1])[:10]}")

INFO - 2026-04-23T13:34:37Z - Selected dataset version: "202311"
INFO - 2026-04-23T13:34:37Z - Selected dataset part: "default"


Dataset dims : {'time': 3379, 'latitude': 72, 'longitude': 108}
Time range   : 2016-07-01 -> 2025-09-30


In [4]:
# Keep JAS months only; convert Kelvin to Celsius
sst_raw  = ds_sst[VAR_NAME]
jas_mask = sst_raw.time.dt.month.isin(MONTHS).values
sst_jas  = (sst_raw.isel(time=jas_mask) - 273.15).rename("sst")

print(f"Loading {jas_mask.sum()} JAS days ...")
sst_jas = sst_jas.load()   # small domain — fits in memory
print(f"SST range : {float(sst_jas.min()):.1f} - {float(sst_jas.max()):.1f} °C")
print(f"First day : {str(sst_jas.time.values[0])[:10]}")
print(f"Last day  : {str(sst_jas.time.values[-1])[:10]}")

Loading 920 JAS days ...
SST range : 20.7 - 30.6 °C
First day : 2016-07-01
Last day  : 2025-09-30


---
## 2. Marine Heatwave Detection

For each pixel and day *t*, compute the **5-day trailing rolling minimum** of SST.  A rolling minimum ≥ 27.5 °C means the pixel has been above the threshold for at least 5 consecutive days — it is *in heatwave*.

Rolling is computed **per year** to avoid spurious multi-year runs across the September→July season boundary.

In [5]:
hw_list = []
for year in YEARS:
    yr_mask = sst_jas.time.dt.year.values == year
    sst_y   = sst_jas.isel(time=yr_mask)
    if len(sst_y.time) == 0:
        continue
    # 5-day trailing rolling minimum (NaN where fewer than 5 valid days)
    roll_min = sst_y.rolling(time=MIN_DAYS, min_periods=MIN_DAYS).min()
    hw       = (roll_min >= THRESHOLD).fillna(False)
    hw_list.append(hw)

heatwave = xr.concat(hw_list, dim="time")   # (n_days, lat, lon) bool
print(f"Heatwave mask shape : {heatwave.shape}")
print(f"Pixel-days in heatwave : {float(heatwave.mean()):.3%}")

Heatwave mask shape : (920, 72, 108)
Pixel-days in heatwave : 0.675%


### 2b. Event labelling

A **basin-scale event** is a contiguous sequence of days on which at least one pixel is in heatwave.  Events are numbered 1, 2, 3, … in chronological order; year boundaries are treated as hard breaks.

In [6]:
# Domain-wide pixel count per day
n_hw   = heatwave.sum(dim=["latitude", "longitude"]).values.astype(float)
times  = pd.DatetimeIndex(heatwave.time.values)
active = n_hw > 0

event_id      = np.zeros(len(times), dtype=int)
current_event = 0
in_evt        = False

for i in range(len(times)):
    if i > 0 and times[i].year != times[i - 1].year:   # hard year break
        in_evt = False
    if active[i]:
        if not in_evt:
            current_event += 1
            in_evt = True
        event_id[i] = current_event
    else:
        in_evt = False

n_events = current_event
print(f"Total heatwave events detected : {n_events}")

Total heatwave events detected : 13


In [8]:
rows = []
for evt in range(1, n_events + 1):
    mask     = event_id == evt
    peak_idx = int(np.argmax(n_hw * mask))   # day with most heatwave pixels
    rows.append({
        "event_id"   : evt,
        "year"       : int(times[mask][0].year),
        "start"      : times[mask][0],
        "end"        : times[mask][-1],
        "duration"   : int(mask.sum()),
        "peak_date"  : times[peak_idx],
        "max_pixels" : int(n_hw[mask].max()),
        "mean_pixels": float(n_hw[mask].mean()),
    })

events_df = pd.DataFrame(rows)
print(events_df.to_string(index=False))

 event_id  year      start        end  duration  peak_date  max_pixels  mean_pixels
        1  2020 2020-08-02 2020-08-03         2 2020-08-03           4     3.000000
        2  2021 2021-08-15 2021-08-18         4 2021-08-16          26    14.000000
        3  2021 2021-08-31 2021-08-31         1 2021-08-31           2     2.000000
        4  2022 2022-07-18 2022-07-19         2 2022-07-18           3     2.000000
        5  2022 2022-07-27 2022-08-15        20 2022-08-07        2840  1469.200000
        6  2022 2022-08-30 2022-08-30         1 2022-08-30          25    25.000000
        7  2022 2022-09-13 2022-09-14         2 2022-09-13           1     1.000000
        8  2023 2023-07-22 2023-08-07        17 2023-08-04        2256  1021.352941
        9  2024 2024-07-22 2024-07-23         2 2024-07-22          31    24.000000
       10  2024 2024-08-02 2024-08-14        13 2024-08-12         216    82.153846
       11  2025 2025-07-07 2025-07-12         6 2025-07-09          67    21

---
## 3. Statistics and Visualisations

### 3a. Number of events per year

In [9]:
events_per_year = (
    events_df.groupby("year")
    .size()
    .reindex(YEARS, fill_value=0)
    .reset_index(name="n_events")
)

events_per_year.hvplot.bar(
    x="year", y="n_events",
    title="Marine Heatwave Events per Year — Gulf of Tunis (JAS)",
    xlabel="Year", ylabel="Number of Events",
    color="coral",
    frame_width=700, 
)

:Bars   [year]   (n_events)

### 3c. Max heatwave pixels per event (coloured by year)

In [10]:
events_df["year_str"] = events_df["year"].astype(str)

events_df.hvplot.scatter(
    x="start", y="max_pixels",
    by="year_str",
    size=100,
    title="Max Heatwave Pixels per Event — Gulf of Tunis (JAS 2016–2025)",
    xlabel="Event Start Date", ylabel="Max pixels in heatwave",
    legend="top_right",
    frame_width=900,
)

:NdOverlay   [year_str]
   :Scatter   [start]   (max_pixels)

### 3d. Event duration distribution

In [11]:
events_df["duration"].hvplot.hist(
    bins=15,
    title="Distribution of Heatwave Event Durations — Gulf of Tunis (JAS 2016–2025)",
    xlabel="Duration (days)", ylabel="Count",
    color="steelblue",
    frame_width=600,
)

:Histogram   [duration]   (Count)

---
## 4. SST Map — 3rd Heatwave Event (Peak Day)

We select the day within event 3 that had the most heatwave pixels and overlay:
- the full SST field (background, `RdYlBu_r`)
- heatwave-flagged pixels (warm `OrRd` overlay)

In [12]:
event3    = events_df[events_df["event_id"] == 5].iloc[0]
peak_date = event3["peak_date"]

print(f"Event 3  : {event3['start'].date()} -> {event3['end'].date()}")
print(f"Duration : {event3['duration']} days")
print(f"Peak day : {peak_date.date()}  ({int(event3['max_pixels'])} heatwave pixels)")

Event 3  : 2022-07-27 -> 2022-08-15
Duration : 20 days
Peak day : 2022-08-07  (2840 heatwave pixels)


In [13]:
sst_peak = sst_jas.sel(time=peak_date, method="nearest")
hw_peak  = heatwave.sel(time=peak_date, method="nearest")

# Full SST background
sst_map = sst_peak.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True, tiles="OSM",
    cmap="RdYlBu_r", 
    clabel="SST (°C)",
    title=f"SST — MHW Event 3 Peak Day  ({peak_date.date()})",
    frame_width=700, alpha=0.75,
)

# Heatwave pixels (SST only where rolling-min >= threshold)
sst_hw = sst_peak.where(hw_peak)
hw_overlay = sst_hw.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="OrRd",
    clabel="Heatwave SST (°C)",
    alpha=0.9,
)

sst_map * hw_overlay

:DynamicMap   []
   :Overlay
      .WMTS.I   :WMTS   [Longitude,Latitude]
      .Image.I  :Image   [longitude,latitude]   (sst)
      .Image.II :Image   [longitude,latitude]   (sst)

---
## 5. Chlorophyll-a Map — Same Date (Sentinel-3A OLCI)

We search the **Planetary Computer STAC API** for a Sentinel-3 OLCI WFR L2 pass closest to the event-3 peak day, then apply the same spatial masking used in the Gulf of Tunis tutorial.

- **Collection:** `sentinel-3-olci-wfr-l2-netcdf`
- **Variable:** `CHL_NN` (Neural Net chlorophyll, mg m⁻³) — best for coastal/turbid waters
- **Resolution:** ~300 m

Lat/lon coordinates live in a separate `geo-coordinates` asset and are merged before masking.

In [36]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Search +/-7 days around peak to maximise chance of a cloud-free pass
t0 = (peak_date - pd.Timedelta(days=7)).date()
t1 = (peak_date + pd.Timedelta(days=7)).date()

search_chl = catalog.search(
    collections=["sentinel-3-olci-wfr-l2-netcdf"],
    bbox=bbox,
    datetime=f"{t0}/{t1}",
)
items_chl = search_chl.item_collection()
print(f"Found {len(items_chl)} OLCI passes in [{t0}, {t1}]")
for it in items_chl:
    print(f"  {it.id}  ({it.datetime.date()})")

Found 20 OLCI passes in [2022-07-31, 2022-08-14]
  S3B_OL_2_WFR_20220814T094815_20220814T095115_0179_069_193_2340  (2022-08-14)
  S3A_OL_2_WFR_20220813T091234_20220813T091534_0180_088_321_2340  (2022-08-13)
  S3A_OL_2_WFR_20220812T093845_20220812T094145_0180_088_307_2340  (2022-08-12)
  S3B_OL_2_WFR_20220812T085937_20220812T090237_0180_069_164_2340  (2022-08-12)
  S3B_OL_2_WFR_20220811T092548_20220811T092848_0179_069_150_2340  (2022-08-11)
  S3B_OL_2_WFR_20220810T095159_20220810T095459_0180_069_136_2340  (2022-08-10)
  S3A_OL_2_WFR_20220809T091619_20220809T091919_0179_088_264_2340  (2022-08-09)
  S3A_OL_2_WFR_20220808T094230_20220808T094530_0179_088_250_2340  (2022-08-08)
  S3B_OL_2_WFR_20220808T090321_20220808T090621_0179_069_107_2340  (2022-08-08)
  S3B_OL_2_WFR_20220807T092932_20220807T093232_0179_069_093_2340  (2022-08-07)
  S3B_OL_2_WFR_20220806T095543_20220806T095843_0179_069_079_2340  (2022-08-06)
  S3A_OL_2_WFR_20220805T092004_20220805T092304_0179_088_207_2340  (2022-08-05)
  S

In [37]:
from datetime import timezone

# Pick the pass closest to the peak day
peak_utc = peak_date.to_pydatetime().replace(tzinfo=timezone.utc)
item_chl  = min(items_chl, key=lambda x: abs((x.datetime - peak_utc).total_seconds()))
print(f"Selected pass : {item_chl.id}")
print(f"Pass datetime : {item_chl.datetime}")

Selected pass : S3B_OL_2_WFR_20220807T092932_20220807T093232_0179_069_093_2340
Pass datetime : 2022-08-07 09:31:02.292251+00:00


In [38]:
def open_asset(item, key):
    """Sign and open a Planetary Computer asset as an xarray Dataset."""
    asset = planetary_computer.sign(item.assets[key])
    return xr.open_dataset(fsspec.open(asset.href).open())

ds_chl = open_asset(item_chl, "chl-nn")
geo    = open_asset(item_chl, "geo-coordinates")

# Merge lat/lon (stored in a separate asset) as 2-D coordinates
ds_chl = ds_chl.assign_coords(
    latitude= (["rows", "columns"], geo["latitude"].values),
    longitude=(["rows", "columns"], geo["longitude"].values),
)
print(ds_chl)

<xarray.Dataset> Size: 478MB
Dimensions:     (rows: 4090, columns: 4865)
Coordinates:
    latitude    (rows, columns) float64 159MB 41.97 41.97 41.97 ... 29.07 29.07
    longitude   (rows, columns) float64 159MB 3.64 3.643 3.646 ... 15.29 15.29
Dimensions without coordinates: rows, columns
Data variables:
    CHL_NN      (rows, columns) float32 80MB ...
    CHL_NN_err  (rows, columns) float32 80MB ...
Attributes: (12/17)
    absolute_orbit_number:  22310
    ac_subsampling_factor:  64
    al_subsampling_factor:  1
    comment:                 
    contact:                ops@eumetsat.int
    creation_time:          2022-08-08T21:18:25Z
    ...                     ...
    references:             S3IPF PDS 004.3 - i2r3 - Product Data Format Spec...
    resolution:             [ 270 294 ]
    source:                 IPF-OL-2 07.02
    start_time:             2022-08-07T09:29:32.330405Z
    stop_time:              2022-08-07T09:32:32.254134Z
    title:                  OLCI Level 2 WATER P

In [39]:
lat_chl = ds_chl["latitude"].values
lon_chl = ds_chl["longitude"].values

mask_chl = (
    (lat_chl >= bbox[1]) & (lat_chl <= bbox[3]) &
    (lon_chl >= bbox[0]) & (lon_chl <= bbox[2])
)

rows_idx, cols_idx = np.where(mask_chl)
print(f"{len(rows_idx)} OLCI pixels inside Gulf of Tunis bbox")

row_min, row_max = rows_idx.min(), rows_idx.max() + 1
col_min, col_max = cols_idx.min(), cols_idx.max() + 1

chl_sub = ds_chl["CHL_NN"].isel(
    rows=slice(row_min, row_max),
    columns=slice(col_min, col_max),
).load()

lat_sub  = lat_chl[row_min:row_max, col_min:col_max]
lon_sub  = lon_chl[row_min:row_max, col_min:col_max]
mask_sub = mask_chl[row_min:row_max, col_min:col_max]

chl_vals = chl_sub.values.copy().astype(float)
chl_vals[~mask_sub] = np.nan

valid_chl = chl_vals[np.isfinite(chl_vals) & (chl_vals > 0)]
print(f"CHL_NN range : {valid_chl.min():.3f} - {valid_chl.max():.3f} mg m-3  |  mean: {valid_chl.mean():.3f}")

68052 OLCI pixels inside Gulf of Tunis bbox
CHL_NN range : 0.016 - 2.000 mg m-3  |  mean: 0.867


In [40]:
df_chl = pd.DataFrame({
    "lat": lat_sub.flatten(),
    "lon": lon_sub.flatten(),
    "chl": chl_vals.flatten(),
})
df_chl = df_chl[df_chl["chl"].notna() & (df_chl["chl"] > 0) & (df_chl["chl"] < 100)]
df_chl["log_chl"] = np.log10(df_chl["chl"])

df_chl.hvplot.points(
    x="lon", y="lat",
    c="log_chl",
    colormap="YlGn",
    clabel="log10 CHL (mg m-3)",
    title=f"Sentinel-3 OLCI Chlorophyll-a — Gulf of Tunis ({item_chl.datetime.date()})",
    xlabel="Longitude", ylabel="Latitude",
    size=6,
    geo=True,
    tiles="OSM",
    frame_width=700,
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [lon,lat]   (log_chl)

---
## 6. Summary

| Step | Dataset | Key variable | Resolution |
|------|---------|-------------|------------|
| MHW detection | CMEMS MED SST L3S MY | `adjusted_sea_surface_temperature` | ~0.01° daily |
| SST map | same | SST (°C) | ~0.01° |
| Chlorophyll-a | Sentinel-3A OLCI WFR L2 (Planetary Computer) | `CHL_NN` | ~300 m |

**Key points:**
- Marine heatwaves here use a **fixed 27.5 °C threshold** — a climatological-percentile-based threshold (Hobday et al. 2016) would be spatially varying and more robust.
- The **rolling-minimum** approach ensures only *sustained* exceedances (≥ 5 consecutive days) are flagged, filtering out brief warm spikes.
- Chlorophyll-a is plotted on a **log₁₀ scale** because coastal values span several orders of magnitude.
- Elevated chlorophyll near the Medjerda River delta in summer often co-occurs with SST anomalies — the spatial patterns can hint at upwelling or nutrient loading.

**Next steps:**
- Replace fixed threshold with the 90th climatological percentile per pixel (Hobday et al. 2016)
- Compute trend in event frequency and total pixel-days over 2016–2025
- Add in-situ SST from Copernicus Marine in-situ service to validate the gridded product
- Overlay surface currents from CMEMS MED Physics to explain spatial SST patterns